#Importing Panda Library

In [ ]:
import pandas as pd
import numpy as np
import requests # to fetch data from URL
import sqlite3


Load Data from Different Data Sources

In [ ]:
# read the data from , Require request library

#access weather data from api, based on Latitude and Longitude
def fetch_weather_data(lat, lon):
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
    response = requests.get(url)
    print(response.json())
    data=response.json()
    weather_data=data["current_weather"]
    return pd.DataFrame([weather_data])


fetch_weather_data(28.6139, 77.2090)


#Using SQLLITE FOR DB Read


In [ ]:
#Read House price data from csv and load into sqlite database
df_houseprices= pd.read_csv("Datasets/Houseprices.csv")

#Create a sqlite database and load the data into it
conn=sqlite3.connect(r"Datasets\housepricesDB.db")
df_houseprices.to_sql("tbl_houseprices", conn, if_exists='replace', index=False)
print("Data loaded into sqlite database successfully.")

#Query the data from sqlite database
query="SELECT * FROM tbl_houseprices where city='Seattle'"
df_query=pd.read_sql_query(query, conn)
df_query

#Date Time functions in Pandas

In [ ]:
#Create sample date data for 5 days
df_dates = pd.DataFrame({
    'Date': pd.date_range(start='2023-01-01', periods=5, freq='D'), 
})

df_dates
#df_dates.dtypes Shows datatype of Date column as datetime64[ns] 

In [ ]:
#extracting year, month, day, weekday from Date column
df_dates['Year'] = df_dates['Date'].dt.year
df_dates['Month'] = df_dates['Date'].dt.month
df_dates['Day'] = df_dates['Date'].dt.day
df_dates['Weekday'] = df_dates['Date'].dt.weekday
df_dates["MonthName"]=df_dates['Date'].dt.month_name()
df_dates["DayName"]=df_dates['Date'].dt.day_name()
df_dates["IsWeekend"]=df_dates['Date'].dt.weekday >4

df_dates

In [ ]:
#Add DF with by hours
date_rng = pd.date_range(start='2023-01-01', periods=24*3, freq='h')
df_hourly = pd.DataFrame(date_rng, columns=['DateTime'])

#Add AM/PM column
df_hourly['AM_PM'] = df_hourly['DateTime'].dt.strftime('%p')

#extract hour from DateTime
df_hourly["Hour"]   = df_hourly['DateTime'].dt.hour

#Add 5 Days to the DateTime column, pd.Timedelta is used to perform date arithmetic
df_hourly['DateTimePlus5Days'] = df_hourly['DateTime'] + pd.Timedelta(days=5)
df_hourly

In [ ]:
#Subtract of 2 date columns to get difference in days
df_hourly['DateDiff'] = (df_hourly['DateTimePlus5Days'] - df_hourly['DateTime']) 
df_hourly

 

In [ ]:
#Convert Date to different timezones, use localize if the Datetime is not having any Tz info else use dt.tz_convert
date_rng_utc = pd.date_range(start='2023-01-01', periods=5, freq='D')
df_utc = pd.DataFrame(date_rng_utc, columns=['DateTime_UTC'])
print(df_utc)
#Convert UTC to US/Pacific timezone
df_utc['DateTime_Pacific'] = df_utc['DateTime_UTC'].dt.tz_localize('US/Pacific')



#LOAD HOUSE PRICE DATA, 
#Resample methods, -- apply aggregate operations on Date


In [ ]:
data_houseprice=pd.read_csv("Datasets/Houseprices.csv")
#data_houseprice.info() 

#Since the date column is in object datatype, we need to convert it to datetime datatype
data_houseprice["date"]=pd.to_datetime(data_houseprice["date"])
#print("\nAfter converting date column to datetime datatype:\n")
#data_houseprice.info()  #Date is converted to datetime64[ns] datatype

#Resample method is applied on the index, so we need to set the date column as index

#Set Date as an Index
data_houseprice.set_index("date", inplace=True)

#Show Daily average house prices
#daily_avg_prices = data_houseprice.resample('D').sum(numeric_only=True)
#daily_avg_prices

#only on one column
daily_sum = data_houseprice.resample('D')['price'].sum(numeric_only=True)
daily_sum

monthly_Sales = data_houseprice.resample('ME')['price'].sum(numeric_only=True)
monthly_Sales


weekly_sales = data_houseprice.resample('W')['price'].sum(numeric_only=True)
weekly_sales